# Process COM Legacy files — v8

This notebook preserves the COM legacy manifest, folder, document-ID and
output conventions from v7, but aligns the extraction behavior with the
latest validated DB-COMP / Curia / Antitrust runs.

Key v8 changes:

- one native-candidate comparison before OCR;
- MinerU only for serious native extraction failures;
- `low_language_plausibility` and repeated-character warnings do not launch OCR;
- MinerU must beat the best native candidate by a meaningful score gain;
- native extraction wins ties and marginal OCR improvements;
- lower quality score alone does not make healthy text `fishy`;
- repeated-character warnings remain diagnostic but do not automatically
  require manual review;
- previous manifests are loaded once, not reread for every document;
- efficient version-aware reruns;
- legacy paths, IDs, manifests and canonical outputs remain unchanged.

Canonical downstream text remains:

`data/processed/com_legacy/<document_id>.txt`


## 1. Configuration and optional dependencies

In [9]:

from __future__ import annotations

import os
import re
import json
import shutil
import hashlib
import subprocess
import unicodedata
import tempfile
from collections import Counter
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

import pandas as pd
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from bs4 import BeautifulSoup, UnicodeDammit
except Exception:
    BeautifulSoup = None
    UnicodeDammit = None

try:
    from charset_normalizer import from_bytes as charset_from_bytes
except Exception:
    charset_from_bytes = None

try:
    from ftfy import fix_text as ftfy_fix_text
except Exception:
    ftfy_fix_text = None

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "data").exists() and (candidate / "output").exists():
        PROJECT_ROOT = candidate
        break

# Override manually if required:
# PROJECT_ROOT = Path("/home/edik/projects/eccjeu").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "com_legacy"
CLEAN_DATA_DIR = DATA_DIR / "processed" / "com_legacy"
CANDIDATE_DATA_DIR = DATA_DIR / "processed" / "com_legacy_candidates"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_legacy"

DOWNLOAD_MANIFEST = OUTPUT_DIR / "download_url_manifest.csv"
FALLBACK_DOWNLOAD_MANIFESTS = [
    PROJECT_ROOT / "output" / "com_legacy" / "download_url_manifest.csv",
    PROJECT_ROOT / "download_url_manifest.csv",
]

CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_FILE_MANIFEST_BASENAME = "com_legacy_clean_file_manifest"
CLEAN_FILE_MANIFEST_PATH = OUTPUT_DIR / f"{CLEAN_FILE_MANIFEST_BASENAME}.csv"
CLEAN_FILE_MANIFEST_XLSX_PATH = OUTPUT_DIR / f"{CLEAN_FILE_MANIFEST_BASENAME}.xlsx"
CANDIDATE_MANIFEST_PATH = OUTPUT_DIR / "com_legacy_extraction_candidate_manifest.csv"
FAILED_FILE_MANIFEST_PATH = OUTPUT_DIR / "com_legacy_failed_cleaning_rows.csv"

# ---------------------------------------------------------------------
# Production defaults
# ---------------------------------------------------------------------
EXTRACTION_VERSION = 8
FORCE_REEXTRACT = True

# Candidate scores are always retained in the candidate manifest.
# Saving every candidate as three separate text files adds substantial I/O,
# so it is disabled by default and can be enabled for debugging.
SAVE_ALL_CANDIDATES = False

RUN_PDFTOTEXT = True
RUN_MINERU = True
RUN_MINERU_FOR_ALL_PDFS = False
MINERU_TIMEOUT_SECONDS = 300

MIN_CHARS_ANY = 100
MIN_PDF_CHARS_GOOD = 1200
MIN_SCORE_ACCEPTABLE = 35.0
MIN_SCORE_OK = 55.0

# OCR is intentionally conservative. A merely imperfect native score should
# not automatically launch a multi-minute MinerU job.
MIN_NATIVE_SCORE_TO_RUN_OCR = 35.0
MIN_MINERU_SCORE_GAIN = 2.5
SERIOUS_OCR_REASONS = {
    "very_short_text",
    "replacement_characters",
    "possible_mojibake",
    "private_use_characters",
}

SERIOUS_REVIEW_REASONS = {
    "very_short_text",
    "replacement_characters",
    "possible_mojibake",
    "private_use_characters",
}

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")

if MINERU_CLI:
    MINERU_COMMAND_TEMPLATE = [
        MINERU_CLI, "-p", "{input_pdf}", "-o", "{output_dir}"
    ]
else:
    MINERU_COMMAND_TEMPLATE = [
        "mineru", "-p", "{input_pdf}", "-o", "{output_dir}"
    ]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DOWNLOAD_MANIFEST:", DOWNLOAD_MANIFEST)
print("Canonical clean directory:", CLEAN_DATA_DIR)
print("Candidate directory:", CANDIDATE_DATA_DIR)
print("PyMuPDF available:", fitz is not None)
print("BeautifulSoup available:", BeautifulSoup is not None)
print("charset-normalizer available:", charset_from_bytes is not None)
print("ftfy available:", ftfy_fix_text is not None)
print("pdftotext:", PDFTOTEXT_CLI)
print("MinerU:", MINERU_CLI)
print("FORCE_REEXTRACT:", FORCE_REEXTRACT)
print("SAVE_ALL_CANDIDATES:", SAVE_ALL_CANDIDATES)


PROJECT_ROOT: /home/edik/projects/eccjeu
DOWNLOAD_MANIFEST: /home/edik/projects/eccjeu/output/com_legacy/download_url_manifest.csv
Canonical clean directory: /home/edik/projects/eccjeu/data/processed/com_legacy
Candidate directory: /home/edik/projects/eccjeu/data/processed/com_legacy_candidates
PyMuPDF available: True
BeautifulSoup available: True
charset-normalizer available: True
ftfy available: True
pdftotext: /usr/bin/pdftotext
MinerU: /home/edik/projects/.venv/bin/mineru
FORCE_REEXTRACT: True
SAVE_ALL_CANDIDATES: False


Optional installation cell. Run only if packages are missing. `pdftotext` is normally installed through the operating system rather than pip.

In [10]:

# Uncomment if needed:
# %pip install -U beautifulsoup4 charset-normalizer ftfy pymupdf openpyxl

# Ubuntu/Debian system package for pdftotext:
# !sudo apt-get update && sudo apt-get install -y poppler-utils


## 2. Load the download manifest and construct the work manifest

In [11]:

def resolve_manifest_path() -> Path:
    if DOWNLOAD_MANIFEST.exists():
        return DOWNLOAD_MANIFEST
    for path in FALLBACK_DOWNLOAD_MANIFESTS:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find the COM legacy download manifest. Expected at: "
        f"{DOWNLOAD_MANIFEST}. Adjust DOWNLOAD_MANIFEST if required."
    )


def as_bool_series(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def first_existing_path(value) -> Optional[Path]:
    if pd.isna(value) or not str(value).strip():
        return None
    for part in [x.strip() for x in str(value).split(";") if x.strip()]:
        path = Path(part)
        if path.exists() and path.is_file():
            return path.resolve()
    return None


def path_exists_from_cell(value) -> bool:
    return first_existing_path(value) is not None


def safe_slug(value: str, max_len: int = 80) -> str:
    value = str(value or "").strip()
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_")
    return value[:max_len] or "document"


def infer_file_format(path: Optional[Path], row_format: str = "") -> str:
    if path is not None and path.suffix:
        extension = path.suffix.lower().lstrip(".")
        if extension in {"htm", "xhtml"}:
            return "html"
        if extension == "text":
            return "txt"
        return extension
    return str(row_format or "unknown").lower()


def make_document_id(row: pd.Series, raw_path: Path) -> str:
    celex = str(row.get("celex") or "").strip()
    source_entry_id = str(row.get("source_entry_id") or "").strip()
    file_format = infer_file_format(raw_path, row.get("file_format", ""))
    base = celex or source_entry_id or raw_path.stem
    digest = hashlib.sha1(
        str(raw_path).encode("utf-8", errors="ignore")
    ).hexdigest()[:10]
    return f"com_legacy__{safe_slug(base)}__{file_format}__{digest}"


manifest_path = resolve_manifest_path()
df = pd.read_csv(manifest_path, low_memory=False)
df.columns = [str(column).strip() for column in df.columns]

required = [
    "source_entry_id",
    "title",
    "download_url",
    "download_path",
    "download_success",
]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Manifest missing expected columns: {missing}")

keep = pd.Series(True, index=df.index)

if "keep_case_candidate" in df.columns:
    keep &= as_bool_series(df["keep_case_candidate"])

# Preserve the legacy exclusion of joined/removed source rows.
joined_removed_pattern = r"\b(joined|removed)\b|\((?:joined|removed)\)"
text_columns = [
    column for column in [
        "title",
        "decision_type",
        "normalized_decision_type",
        "decision_family",
        "source_entry_id",
    ]
    if column in df.columns
]
joined_removed = pd.Series(False, index=df.index)
for column in text_columns:
    joined_removed |= (
        df[column]
        .astype(str)
        .str.contains(
            joined_removed_pattern,
            case=False,
            regex=True,
            na=False,
        )
    )
keep &= ~joined_removed

download_success = as_bool_series(df["download_success"])
file_exists_now = df["download_path"].apply(path_exists_from_cell)
has_download_object = (
    df["download_url"].notna() | df["download_path"].notna()
)
keep &= has_download_object & (download_success | file_exists_now)

work = df.loc[keep].copy()
work["download_success_flag"] = download_success.loc[work.index].values
work["raw_file_exists"] = file_exists_now.loc[work.index].values

SUPPORTED_EXTENSIONS = {
    ".pdf", ".html", ".htm", ".xhtml", ".txt", ".text"
}

records = []
for _, row in work.iterrows():
    raw_path = first_existing_path(row.get("download_path"))
    file_format = infer_file_format(
        raw_path,
        row.get("file_format", ""),
    )
    document_id = make_document_id(row, raw_path) if raw_path else ""

    records.append({
        "document_id": document_id,
        "source_entry_id": row.get("source_entry_id", ""),
        "title": row.get("title", ""),
        "decision_date": row.get("decision_date", ""),
        "case_numbers": row.get("case_numbers", ""),
        "celex": row.get("celex", ""),
        "file_format": file_format,
        "download_url": row.get("download_url", ""),
        "download_success": bool(
            row.get("download_success_flag", False)
        ),
        "download_status": row.get("download_status", ""),
        "raw_file_exists": raw_path is not None,
        "raw_file_path": (
            str(raw_path)
            if raw_path
            else str(row.get("download_path", ""))
        ),
    })

work_manifest = pd.DataFrame(records)
work_manifest = (
    work_manifest
    .drop_duplicates(
        subset=["raw_file_path", "file_format"],
        keep="first",
    )
    .reset_index(drop=True)
)

work_manifest["supported_file_type"] = (
    work_manifest["raw_file_path"].apply(
        lambda value: (
            Path(str(value)).suffix.lower() in SUPPORTED_EXTENSIONS
            if str(value).strip()
            else False
        )
    )
)
work_manifest["cleaning_target"] = (
    work_manifest["raw_file_exists"]
    & work_manifest["supported_file_type"]
)

print("Loaded:", manifest_path)
print("Rows in source manifest:", len(df))
print("Rows in work manifest:", len(work_manifest))
print("Cleaning targets:", int(work_manifest["cleaning_target"].sum()))
print("Excluded joined/removed rows:", int(joined_removed.sum()))
display(work_manifest.head(20))


Loaded: /home/edik/projects/eccjeu/output/com_legacy/download_url_manifest.csv
Rows in source manifest: 1781
Rows in work manifest: 1505
Cleaning targets: 1505
Excluded joined/removed rows: 0


/tmp/ipykernel_123225/3964587350.py:101: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_123225/3964587350.py:101: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_123225/3964587350.py:101: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_123225/3964587350.py:101: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_123225/3964587350.py:101: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,document_id,source_entry_id,title,decision_date,case_numbers,celex,file_format,download_url,download_success,download_status,raw_file_exists,raw_file_path,supported_file_type,cleaning_target
0,com_legacy__31964D0599__html__f1021f1378,com_legacy_1964_line_2,Deca,22.10.1964,IV/71,31964D0599,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
1,com_legacy__31964D0599__pdf__1e622a12d6,com_legacy_1964_line_2,Deca,22.10.1964,IV/71,31964D0599,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
2,com_legacy__31964D0566__html__81fa0f8340,com_legacy_1964_line_5,Grundig-Consten,23.09.1964,IV/3344; IV/4,31964D0566,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
3,com_legacy__31964D0566__pdf__df8a10f45c,com_legacy_1964_line_5,Grundig-Consten,23.09.1964,IV/3344; IV/4,31964D0566,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
4,com_legacy__31964D0502__html__0e763dbecc,com_legacy_1964_line_8,Nicholas Freres + Vitapro,30.07.1964,IV/95,31964D0502,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
5,com_legacy__31964D0502__pdf__f50a290302,com_legacy_1964_line_8,Nicholas Freres + Vitapro,30.07.1964,IV/95,31964D0502,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
6,com_legacy__31964D0344__html__9d411975d7,com_legacy_1964_line_11,Bendix + Mertens and Straat,01.06.1964,IV/12868,31964D0344,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
7,com_legacy__31964D0344__pdf__944f0fa8a5,com_legacy_1964_line_11,Bendix + Mertens and Straat,01.06.1964,IV/12868,31964D0344,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
8,com_legacy__31964D0233__html__a25c04d101,com_legacy_1964_line_14,Grosfillex + Fillistorf,11.03.1964,IV/61,31964D0233,html,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True
9,com_legacy__31964D0233__pdf__0f34c27802,com_legacy_1964_line_14,Grosfillex + Fillistorf,11.03.1964,IV/61,31964D0233,pdf,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,already_downloaded,True,/home/edik/projects/eccjeu/data/raw/com_legacy...,True,True


## 3. Text decoding, cleaning, diagnostics, and quality scoring

In [12]:

MOJIBAKE_PATTERNS = [
    "Ã", "Â", "â€™", "â€œ", "â€", "ðŸ", "�",
]

LEGAL_TERMS = {
    "en": ["commission", "decision", "court", "article", "applicant", "undertaking", "competition"],
    "de": ["kommission", "entscheidung", "gericht", "artikel", "kläger", "unternehmen", "wettbewerb"],
    "fr": ["commission", "décision", "cour", "article", "requérant", "entreprise", "concurrence"],
    "it": ["commissione", "decisione", "corte", "articolo", "ricorrente", "impresa", "concorrenza"],
}

COMMON_WORDS = {
    "en": ["the", "of", "and", "to", "in", "that", "for", "on", "with"],
    "de": ["der", "die", "das", "und", "von", "zu", "in", "für", "mit"],
    "fr": ["de", "la", "le", "et", "des", "les", "du", "pour", "dans"],
    "it": ["di", "la", "il", "e", "del", "della", "per", "in", "con"],
}


def read_file_bytes(path: Path) -> bytes:
    return path.read_bytes()


def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def score_decoded_text(text: str) -> float:
    if not text:
        return -100.0
    n = len(text)
    replacement_ratio = text.count("�") / max(n, 1)
    control_ratio = sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)
    mojibake_ratio = sum(text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in text) / max(n, 1)
    return (
        printable_ratio * 30
        - replacement_ratio * 300
        - control_ratio * 200
        - mojibake_ratio * 180
    )


def extract_declared_html_encoding(data: bytes) -> Optional[str]:
    head = data[:8192]
    ascii_head = head.decode("ascii", errors="ignore")
    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, ascii_head, flags=re.I)
        if match:
            return match.group(1).strip()
    return None


def decode_bytes_smart(data: bytes, is_html: bool = False) -> Tuple[str, Dict[str, Any]]:
    candidates: List[Tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(("utf-8-sig", "bom", data.decode("utf-8-sig", errors="replace")))
    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(("utf-16", "bom", data.decode("utf-16", errors="replace")))

    declared = extract_declared_html_encoding(data) if is_html else None
    if declared:
        try:
            candidates.append((declared, "declared", data.decode(declared, errors="replace")))
        except Exception:
            pass

    if is_html and UnicodeDammit is not None:
        try:
            dammit = UnicodeDammit(data, is_html=True, smart_quotes_to=None)
            if dammit.unicode_markup:
                candidates.append((
                    dammit.original_encoding or "unknown",
                    "unicode_dammit",
                    dammit.unicode_markup,
                ))
        except Exception:
            pass

    try:
        candidates.append(("utf-8", "strict_utf8", data.decode("utf-8", errors="strict")))
    except UnicodeDecodeError:
        pass

    if charset_from_bytes is not None:
        try:
            matches = charset_from_bytes(data)
            for match in list(matches)[:4]:
                text = str(match)
                candidates.append((
                    getattr(match, "encoding", None) or "unknown",
                    "charset_normalizer",
                    text,
                ))
        except Exception:
            pass

    for encoding in ["cp1252", "latin-1"]:
        try:
            candidates.append((encoding, "fallback", data.decode(encoding, errors="replace")))
        except Exception:
            pass

    if not candidates:
        candidates.append(("utf-8", "last_resort", data.decode("utf-8", errors="replace")))

    unique = {}
    for enc, source, text in candidates:
        key = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()
        unique.setdefault(key, (enc, source, text))

    scored = []
    for enc, source, text in unique.values():
        base_score = score_decoded_text(text)
        repaired = None
        repaired_score = None

        if ftfy_fix_text is not None:
            try:
                repaired = ftfy_fix_text(text)
                repaired_score = score_decoded_text(repaired)
            except Exception:
                repaired = None

        use_repaired = (
            repaired is not None
            and repaired != text
            and repaired_score is not None
            and repaired_score > base_score + 0.5
        )
        final_text = repaired if use_repaired else text
        final_score = repaired_score if use_repaired else base_score

        scored.append({
            "encoding": enc,
            "encoding_source": source,
            "text": final_text,
            "decode_score": float(final_score),
            "ftfy_applied": bool(use_repaired),
            "declared_encoding": declared or "",
        })

    best = max(scored, key=lambda x: x["decode_score"])
    meta = {k: v for k, v in best.items() if k != "text"}
    meta["decode_candidate_count"] = len(scored)
    return normalize_unicode(best["text"]), meta


def clean_text_readable(text: str) -> str:
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def clean_text_for_regex_and_llm(text: str) -> str:
    text = clean_text_readable(text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def repeated_character_ratio(text: str) -> float:
    if not text:
        return 1.0
    repeated = sum(len(m.group(0)) for m in re.finditer(r"(.)\1{5,}", text, flags=re.S))
    return repeated / max(len(text), 1)


def duplicate_line_ratio(text: str) -> float:
    lines = [
        re.sub(r"\s+", " ", line).strip().lower()
        for line in text.splitlines()
        if len(re.sub(r"\s+", " ", line).strip()) >= 20
    ]
    if not lines:
        return 0.0
    counts = Counter(lines)
    duplicate_instances = sum(count - 1 for count in counts.values() if count > 1)
    return duplicate_instances / max(len(lines), 1)


def private_use_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(
        0xE000 <= ord(ch) <= 0xF8FF
        for ch in text
    ) / max(len(text), 1)


def token_diagnostics(text: str) -> Dict[str, float]:
    tokens = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text, flags=re.UNICODE)
    if not tokens:
        return {
            "one_char_token_ratio": 1.0,
            "very_long_token_ratio": 1.0,
            "no_vowel_token_ratio": 1.0,
            "mixed_alnum_token_ratio": 0.0,
        }

    vowels = set("aeiouyäöüàâæçéèêëîïôœùûüÿ")
    one_char = sum(len(t) == 1 for t in tokens)
    very_long = sum(len(t) > 30 for t in tokens)
    no_vowel = sum(
        len(t) >= 5 and not any(ch.lower() in vowels for ch in t)
        for t in tokens
    )
    mixed = sum(any(ch.isalpha() for ch in t) and any(ch.isdigit() for ch in t) for t in tokens)

    n = len(tokens)
    return {
        "one_char_token_ratio": one_char / n,
        "very_long_token_ratio": very_long / n,
        "no_vowel_token_ratio": no_vowel / n,
        "mixed_alnum_token_ratio": mixed / n,
    }


def language_plausibility(text: str) -> Tuple[float, str]:
    words = re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+\b", text.lower(), flags=re.UNICODE)
    if not words:
        return 0.0, ""

    counts = Counter(words)
    scores = {}
    for lang in COMMON_WORDS:
        common_hits = sum(counts[w] for w in COMMON_WORDS[lang])
        legal_hits = sum(counts[w] for w in LEGAL_TERMS[lang])
        scores[lang] = min(1.0, (common_hits * 0.5 + legal_hits * 2.0) / max(len(words) * 0.01, 1))

    best_lang = max(scores, key=scores.get)
    return float(scores[best_lang]), best_lang


def text_quality_profile(raw_text: str, clean_text: str, meta: Dict[str, Any], file_format: str) -> Dict[str, Any]:
    raw_text = raw_text or ""
    clean_text = clean_text or ""
    n = len(clean_text)

    alpha_ratio = sum(ch.isalpha() for ch in clean_text) / max(n, 1)
    digit_ratio = sum(ch.isdigit() for ch in clean_text) / max(n, 1)
    printable_ratio = sum(ch.isprintable() or ch in "\n\t" for ch in clean_text) / max(n, 1)
    replacement_ratio = clean_text.count("�") / max(n, 1)
    mojibake_ratio = sum(clean_text.count(p) for p in MOJIBAKE_PATTERNS) / max(n, 1)
    control_ratio = sum(
        1 for ch in clean_text
        if unicodedata.category(ch).startswith("C") and ch not in "\n\t"
    ) / max(n, 1)

    token_stats = token_diagnostics(clean_text)
    lang_score, detected_language = language_plausibility(clean_text)
    repeat_ratio = repeated_character_ratio(clean_text)
    duplicate_ratio = duplicate_line_ratio(clean_text)
    pua_ratio = private_use_ratio(clean_text)

    length_component = min(25.0, n / 800.0)
    score = (
        length_component
        + printable_ratio * 15
        + min(alpha_ratio / 0.65, 1.0) * 10
        + lang_score * 20
        - replacement_ratio * 250
        - mojibake_ratio * 180
        - control_ratio * 250
        - pua_ratio * 250
        - repeat_ratio * 100
        - duplicate_ratio * 20
        - token_stats["one_char_token_ratio"] * 18
        - token_stats["very_long_token_ratio"] * 80
        - token_stats["no_vowel_token_ratio"] * 18
    )

    reasons = []
    if n < MIN_CHARS_ANY:
        reasons.append("very_short_text")
    elif file_format == "pdf" and n < MIN_PDF_CHARS_GOOD:
        reasons.append("short_pdf_text")
    if replacement_ratio > 0.002:
        reasons.append("replacement_characters")
    if mojibake_ratio > 0.0005:
        reasons.append("possible_mojibake")
    if pua_ratio > 0.0005:
        reasons.append("private_use_characters")
    if repeat_ratio > 0.01:
        reasons.append("repeated_character_garbage")
    if duplicate_ratio > 0.20:
        reasons.append("many_duplicate_lines")
    if token_stats["one_char_token_ratio"] > 0.30:
        reasons.append("many_one_character_tokens")
    if token_stats["very_long_token_ratio"] > 0.01:
        reasons.append("many_very_long_tokens")
    if lang_score < 0.10 and n > 1000:
        reasons.append("low_language_plausibility")

    if n < MIN_CHARS_ANY or score < MIN_SCORE_ACCEPTABLE:
        quality_flag = "failed"
    elif reasons:
        quality_flag = "fishy"
    else:
        quality_flag = "ok"

    return {
        "quality_score": round(float(score), 3),
        "quality_flag": quality_flag,
        "quality_reasons": "|".join(reasons),
        "n_chars_raw": len(raw_text),
        "n_chars_clean": n,
        "n_pages": meta.get("n_pages"),
        "alpha_ratio": round(alpha_ratio, 6),
        "digit_ratio": round(digit_ratio, 6),
        "printable_ratio": round(printable_ratio, 6),
        "replacement_ratio": round(replacement_ratio, 8),
        "mojibake_ratio": round(mojibake_ratio, 8),
        "private_use_ratio": round(pua_ratio, 8),
        "repeated_character_ratio": round(repeat_ratio, 8),
        "duplicate_line_ratio": round(duplicate_ratio, 8),
        "language_score": round(lang_score, 6),
        "detected_language": detected_language,
        **{k: round(v, 8) for k, v in token_stats.items()},
    }


## 4. HTML, TXT, PDF, Poppler, and MinerU candidate extraction

In [13]:

HTML_REMOVE_SELECTORS = [
    "script", "style", "noscript", "nav", "footer", "header", "form", "aside",
    "[class*='cookie']", "[id*='cookie']", "[class*='share']",
    "[class*='language']", "[aria-label*='language']",
    "[class*='pagination']", "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main", "article", "#document1", "#TexteOnly", "#text",
    ".document-content", ".document", ".content", "body",
]


def html_container_score(tag) -> float:
    text = tag.get_text(" ", strip=True)
    if not text:
        return -100.0

    links = tag.find_all("a")
    link_text_chars = sum(len(a.get_text(" ", strip=True)) for a in links)
    link_density = link_text_chars / max(len(text), 1)
    paragraphs = len(tag.find_all(["p", "div", "li", "table"]))
    lang_score, _ = language_plausibility(text)

    return (
        min(len(text), 150_000) / 1500
        + min(paragraphs, 200) * 0.08
        + lang_score * 15
        - link_density * 40
    )


def extract_text_from_html(path: Path) -> Tuple[str, Dict[str, Any]]:
    data = read_file_bytes(path)
    html, decode_meta = decode_bytes_smart(data, is_html=True)

    if BeautifulSoup is None:
        text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.I | re.S)
        text = re.sub(r"<[^>]+>", " ", text)
        return normalize_unicode(text), {
            "method": "html_regex",
            "n_pages": None,
            **decode_meta,
        }

    soup = BeautifulSoup(html, "html.parser")

    for selector in HTML_REMOVE_SELECTORS:
        try:
            for tag in soup.select(selector):
                tag.decompose()
        except Exception:
            pass

    candidates = []
    for selector in HTML_CANDIDATE_SELECTORS:
        try:
            for tag in soup.select(selector):
                text = tag.get_text("\n", strip=True)
                if len(text) >= 100:
                    candidates.append((html_container_score(tag), selector, text))
        except Exception:
            pass

    if candidates:
        _, selected_selector, text = max(candidates, key=lambda x: x[0])
    else:
        selected_selector = "document"
        text = soup.get_text("\n", strip=True)

    return normalize_unicode(text), {
        "method": "html_bs4_best_container",
        "n_pages": None,
        "html_selected_container": selected_selector,
        "html_candidate_count": len(candidates),
        **decode_meta,
    }


def extract_text_from_plain(path: Path) -> Tuple[str, Dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(read_file_bytes(path), is_html=False)
    return text, {"method": "plain_text_smart_decode", "n_pages": None, **decode_meta}


def extract_pdf_pymupdf_text(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            text = page.get_text("text", sort=True) or ""
            chunks.append(f"\n\n[page {page_no}]\n{text}")
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_text_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pymupdf_blocks(path: Path) -> Tuple[str, Dict[str, Any]]:
    if fitz is None:
        raise RuntimeError("PyMuPDF is not installed.")

    chunks = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks", sort=True) or []
            block_texts = []
            for block in blocks:
                if len(block) >= 5:
                    text = str(block[4] or "").strip()
                    if text:
                        block_texts.append(text)
            chunks.append(f"\n\n[page {page_no}]\n" + "\n\n".join(block_texts))
        n_pages = len(doc)

    return normalize_unicode("".join(chunks)), {
        "method": "pdf_pymupdf_blocks_sorted",
        "n_pages": n_pages,
    }


def extract_pdf_pdftotext(path: Path) -> Tuple[str, Dict[str, Any]]:
    if not PDFTOTEXT_CLI:
        raise RuntimeError("pdftotext is not installed or not on PATH.")

    with tempfile.TemporaryDirectory(prefix="pdftotext_") as tmpdir:
        out_txt = Path(tmpdir) / "output.txt"
        cmd = [PDFTOTEXT_CLI, "-layout", "-enc", "UTF-8", str(path), str(out_txt)]
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if proc.returncode != 0:
            raise RuntimeError(proc.stderr[-2000:] or "pdftotext failed")
        text = out_txt.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_pdftotext_layout",
        "n_pages": n_pages,
    }


def format_mineru_command(input_pdf: Path, output_dir: Path) -> List[str]:
    return [
        part.format(input_pdf=str(input_pdf), output_dir=str(output_dir))
        for part in MINERU_COMMAND_TEMPLATE
    ]


def run_mineru(input_pdf: Path, output_dir: Path) -> Dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = format_mineru_command(input_pdf, output_dir)

    try:
        env = os.environ.copy()
        env.setdefault("ORT_LOG_SEVERITY_LEVEL", "3")
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=MINERU_TIMEOUT_SECONDS,
            env=env,
        )
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "cmd": " ".join(cmd),
            "stdout_tail": proc.stdout[-2000:],
            "stderr_tail": proc.stderr[-2000:],
        }
    except Exception as exc:
        return {
            "ok": False,
            "returncode": None,
            "cmd": " ".join(cmd),
            "error": repr(exc),
        }


def find_mineru_text(output_dir: Path) -> Optional[Path]:
    candidates = []
    for pattern in ["**/*.md", "**/*.txt"]:
        candidates.extend(output_dir.glob(pattern))
    candidates = [p for p in candidates if p.is_file() and p.stat().st_size > 0]
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_size)


def extract_pdf_mineru(path: Path) -> Tuple[str, Dict[str, Any]]:
    with tempfile.TemporaryDirectory(prefix="mineru_") as tmpdir:
        out_dir = Path(tmpdir)
        result = run_mineru(path, out_dir)
        if not result.get("ok"):
            raise RuntimeError(result.get("error") or result.get("stderr_tail") or "MinerU failed")

        text_file = find_mineru_text(out_dir)
        if text_file is None:
            raise RuntimeError("MinerU completed but produced no non-empty markdown/text file.")

        text = text_file.read_text(encoding="utf-8", errors="replace")

    n_pages = None
    if fitz is not None:
        try:
            with fitz.open(path) as doc:
                n_pages = len(doc)
        except Exception:
            pass

    return normalize_unicode(text), {
        "method": "pdf_mineru",
        "n_pages": n_pages,
    }


def make_candidate(
    document_id: str,
    method: str,
    raw_text: str,
    meta: Dict[str, Any],
    file_format: str,
) -> Dict[str, Any]:
    readable_text = clean_text_readable(raw_text)
    regex_text = clean_text_for_regex_and_llm(raw_text)
    quality = text_quality_profile(raw_text, regex_text, meta, file_format)

    return {
        "document_id": document_id,
        "candidate_method": method,
        "raw_text": raw_text,
        "readable_text": readable_text,
        "clean_text": regex_text,
        "meta": meta,
        **quality,
    }


def generate_candidates(row: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], List[str]]:
    path = Path(str(row["raw_file_path"]))
    file_format = row.get("file_format", "")
    document_id = row["document_id"]
    candidates = []
    errors = []

    def attempt(method_name, extractor):
        try:
            raw_text, meta = extractor(path)
            candidates.append(make_candidate(
                document_id=document_id,
                method=method_name,
                raw_text=raw_text,
                meta=meta,
                file_format=file_format,
            ))
        except Exception as exc:
            errors.append(f"{method_name}: {repr(exc)}")

    if file_format == "html":
        attempt("html_best_container", extract_text_from_html)
    elif file_format == "txt":
        attempt("plain_text_smart_decode", extract_text_from_plain)
    elif file_format == "pdf":
        attempt("pdf_pymupdf_text_sorted", extract_pdf_pymupdf_text)
        attempt("pdf_pymupdf_blocks_sorted", extract_pdf_pymupdf_blocks)

        if RUN_PDFTOTEXT and PDFTOTEXT_CLI:
            attempt("pdf_pdftotext_layout", extract_pdf_pdftotext)

        native_candidates = [
            candidate
            for candidate in candidates
            if candidate.get("candidate_method") != "pdf_mineru"
        ]
        best_native = max(
            native_candidates,
            key=candidate_preference,
            default=None,
        )

        best_native_score = (
            float(best_native.get("quality_score", -999))
            if best_native
            else -999
        )
        best_native_reasons = {
            reason
            for reason in str(
                best_native.get("quality_reasons", "")
                if best_native
                else ""
            ).split("|")
            if reason
        }

        should_run_ocr = (
            RUN_MINERU
            and MINERU_CLI is not None
            and (
                RUN_MINERU_FOR_ALL_PDFS
                or best_native is None
                or best_native.get("quality_flag") == "failed"
                or best_native_score < MIN_NATIVE_SCORE_TO_RUN_OCR
                or bool(best_native_reasons & SERIOUS_OCR_REASONS)
            )
        )

        if should_run_ocr:
            attempt("pdf_mineru", extract_pdf_mineru)
    else:
        errors.append(f"unsupported_file_format: {file_format}")

    return candidates, errors


## 5. Candidate selection, canonical output, and manifests

In [14]:

def candidate_preference(candidate: Dict[str, Any]) -> Tuple[float, int]:
    # Quality score dominates. This tie-breaker prefers less destructive/native methods.
    method_order = {
        "html_best_container": 5,
        "plain_text_smart_decode": 5,
        "pdf_pymupdf_text_sorted": 4,
        "pdf_pdftotext_layout": 3,
        "pdf_pymupdf_blocks_sorted": 2,
        "pdf_mineru": 1,
    }
    return (
        float(candidate.get("quality_score", -999)),
        method_order.get(candidate.get("candidate_method", ""), 0),
    )


def select_best_candidate(
    candidates: List[Dict[str, Any]],
) -> Tuple[Optional[Dict[str, Any]], str]:
    if not candidates:
        return None, "no_candidates"

    native_candidates = [
        candidate
        for candidate in candidates
        if candidate.get("candidate_method") != "pdf_mineru"
    ]
    mineru_candidate = next(
        (
            candidate
            for candidate in candidates
            if candidate.get("candidate_method") == "pdf_mineru"
        ),
        None,
    )

    best_native = max(
        native_candidates,
        key=candidate_preference,
        default=None,
    )

    if mineru_candidate is None:
        selected = max(
            candidates,
            key=candidate_preference,
        )
        return selected, "best_available_candidate"

    if best_native is None:
        return mineru_candidate, "native_missing_mineru_available"

    native_score = float(
        best_native.get("quality_score", -999)
    )
    mineru_score = float(
        mineru_candidate.get("quality_score", -999)
    )

    if (
        best_native.get("quality_flag") == "failed"
        and mineru_candidate.get("quality_flag") != "failed"
    ):
        return mineru_candidate, "native_failed_mineru_valid"

    gain = mineru_score - native_score

    if gain >= MIN_MINERU_SCORE_GAIN:
        return (
            mineru_candidate,
            f"mineru_gain_{gain:.3f}_meets_threshold",
        )

    return (
        best_native,
        f"mineru_gain_{gain:.3f}_below_threshold",
    )


def save_candidate_files(document_id: str, candidate: Dict[str, Any]) -> Dict[str, str]:
    method = safe_slug(candidate["candidate_method"], max_len=50)
    doc_dir = CANDIDATE_DATA_DIR / document_id
    doc_dir.mkdir(parents=True, exist_ok=True)

    raw_path = doc_dir / f"{method}__raw.txt"
    readable_path = doc_dir / f"{method}__readable.txt"
    regex_path = doc_dir / f"{method}__regex.txt"

    raw_path.write_text(candidate.get("raw_text", ""), encoding="utf-8")
    readable_path.write_text(candidate.get("readable_text", ""), encoding="utf-8")
    regex_path.write_text(candidate.get("clean_text", ""), encoding="utf-8")

    return {
        "candidate_raw_path": str(raw_path),
        "candidate_readable_path": str(readable_path),
        "candidate_regex_path": str(regex_path),
    }


def write_markdown(
    row: Dict[str, Any],
    clean_text: str,
    markdown_path: Path,
    selected_candidate: Dict[str, Any],
) -> None:
    meta = {
        "document_id": row.get("document_id"),
        "source": "com_legacy",
        "title": row.get("title"),
        "decision_date": row.get("decision_date"),
        "case_numbers": row.get("case_numbers"),
        "source_entry_id": row.get("source_entry_id"),
        "celex": row.get("celex"),
        "file_format": row.get("file_format"),
        "raw_file_path": row.get("raw_file_path"),
        "download_url": row.get("download_url"),
        "extraction_version": EXTRACTION_VERSION,
        "selected_method": selected_candidate.get("candidate_method"),
        "quality_score": selected_candidate.get("quality_score"),
        "quality_flag": selected_candidate.get("quality_flag"),
        "quality_reasons": selected_candidate.get("quality_reasons"),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }
    header = (
        "---\n"
        + "\n".join(f"{key}: {json.dumps(value, ensure_ascii=False)}" for key, value in meta.items())
        + "\n---\n\n"
    )
    markdown_path.write_text(header + clean_text, encoding="utf-8")


def load_previous_manifests():
    previous_clean = pd.DataFrame()
    previous_candidates = pd.DataFrame()

    if (
        CLEAN_FILE_MANIFEST_PATH.exists()
        and CLEAN_FILE_MANIFEST_PATH.stat().st_size > 0
    ):
        try:
            previous_clean = pd.read_csv(
                CLEAN_FILE_MANIFEST_PATH,
                low_memory=False,
            )
        except pd.errors.EmptyDataError:
            previous_clean = pd.DataFrame()

    if (
        CANDIDATE_MANIFEST_PATH.exists()
        and CANDIDATE_MANIFEST_PATH.stat().st_size > 0
    ):
        try:
            previous_candidates = pd.read_csv(
                CANDIDATE_MANIFEST_PATH,
                low_memory=False,
            )
        except pd.errors.EmptyDataError:
            previous_candidates = pd.DataFrame()

    clean_lookup = {}
    if not previous_clean.empty and "document_id" in previous_clean.columns:
        for _, row in previous_clean.iterrows():
            clean_lookup[str(row["document_id"])] = row.to_dict()

    candidate_lookup = {}
    if (
        not previous_candidates.empty
        and "document_id" in previous_candidates.columns
    ):
        for document_id, group in previous_candidates.groupby(
            previous_candidates["document_id"].astype(str)
        ):
            candidate_lookup[str(document_id)] = group.to_dict("records")

    return clean_lookup, candidate_lookup


PREVIOUS_CLEAN_LOOKUP, PREVIOUS_CANDIDATE_LOOKUP = (
    load_previous_manifests()
)


def existing_output_is_current(document_id: str) -> bool:
    if FORCE_REEXTRACT:
        return False

    row = PREVIOUS_CLEAN_LOOKUP.get(str(document_id))
    if not row:
        return False

    try:
        version_ok = (
            int(float(row.get("extraction_version", 0)))
            == EXTRACTION_VERSION
        )
    except Exception:
        version_ok = False

    required_paths = [
        row.get("clean_text_path", ""),
        row.get("readable_text_path", ""),
        row.get("markdown_path", ""),
    ]

    paths_ok = all(
        Path(str(path)).exists()
        and Path(str(path)).stat().st_size > 0
        for path in required_paths
        if str(path).strip()
    )

    return version_ok and paths_ok


def process_one(row: Dict[str, Any]) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    document_id = row["document_id"]
    clean_text_path = CLEAN_DATA_DIR / f"{document_id}.txt"
    readable_text_path = CLEAN_DATA_DIR / f"{document_id}__readable.txt"
    markdown_path = CLEAN_DATA_DIR / f"{document_id}.md"

    base = {
        **row,
        "processed_at": datetime.now().isoformat(timespec="seconds"),
        "extraction_version": EXTRACTION_VERSION,
        "clean_text_path": str(clean_text_path),
        "readable_text_path": str(readable_text_path),
        "markdown_path": str(markdown_path),
    }

    if not row.get("cleaning_target", False):
        result = {
            **base,
            "clean_success": False,
            "selected_method": "",
            "extraction_engine": "",
            "quality_flag": "skipped",
            "quality_score": None,
            "n_chars_clean": 0,
            "candidate_count": 0,
            "selection_reason": "not_a_cleaning_target",
            "error": "not_a_cleaning_target",
        }
        return result, []

    if existing_output_is_current(document_id):
        hit = dict(PREVIOUS_CLEAN_LOOKUP[str(document_id)])
        hit["processed_at"] = datetime.now().isoformat(
            timespec="seconds"
        )
        hit["selection_reason"] = "reused_current_v8_output"
        return (
            hit,
            PREVIOUS_CANDIDATE_LOOKUP.get(
                str(document_id),
                [],
            ),
        )

    candidates, errors = generate_candidates(row)

    candidate_rows = []
    for candidate in candidates:
        paths = {}
        if SAVE_ALL_CANDIDATES:
            paths = save_candidate_files(document_id, candidate)

        candidate_rows.append({
            "document_id": document_id,
            "raw_file_path": row.get("raw_file_path"),
            "file_format": row.get("file_format"),
            "candidate_method": candidate.get("candidate_method"),
            "quality_score": candidate.get("quality_score"),
            "quality_flag": candidate.get("quality_flag"),
            "quality_reasons": candidate.get("quality_reasons"),
            "n_chars_raw": candidate.get("n_chars_raw"),
            "n_chars_clean": candidate.get("n_chars_clean"),
            "n_pages": candidate.get("n_pages"),
            "detected_language": candidate.get("detected_language"),
            "language_score": candidate.get("language_score"),
            "replacement_ratio": candidate.get("replacement_ratio"),
            "mojibake_ratio": candidate.get("mojibake_ratio"),
            "private_use_ratio": candidate.get("private_use_ratio"),
            "repeated_character_ratio": candidate.get("repeated_character_ratio"),
            "duplicate_line_ratio": candidate.get("duplicate_line_ratio"),
            "one_char_token_ratio": candidate.get("one_char_token_ratio"),
            "very_long_token_ratio": candidate.get("very_long_token_ratio"),
            "no_vowel_token_ratio": candidate.get("no_vowel_token_ratio"),
            "meta_json": json.dumps(candidate.get("meta", {}), ensure_ascii=False),
            **paths,
        })

    if not candidates:
        result = {
            **base,
            "clean_success": False,
            "selected_method": "",
            "extraction_engine": "",
            "quality_flag": "failed",
            "quality_score": None,
            "n_chars_clean": 0,
            "candidate_count": 0,
            "selection_reason": "no_candidate_succeeded",
            "error": " | ".join(errors),
        }
        return result, candidate_rows

    selected, selection_reason = select_best_candidate(
        candidates
    )
    if selected is None:
        result = {
            **base,
            "clean_success": False,
            "selected_method": "",
            "extraction_engine": "",
            "quality_flag": "failed",
            "quality_score": None,
            "needs_manual_review": True,
            "n_chars_clean": 0,
            "candidate_count": len(candidates),
            "selection_reason": selection_reason,
            "error": "no_candidate_succeeded",
        }
        return result, candidate_rows
    clean_text = selected.get("clean_text", "")
    readable_text = selected.get("readable_text", "")

    clean_text_path.write_text(clean_text, encoding="utf-8")
    readable_text_path.write_text(readable_text, encoding="utf-8")
    write_markdown(row, clean_text, markdown_path, selected)

    score_listing = "; ".join(
        f"{c['candidate_method']}={c['quality_score']:.2f}"
        for c in sorted(candidates, key=candidate_preference, reverse=True)
    )

    clean_success = bool(
        clean_text
        and selected.get("quality_flag") != "failed"
        and selected.get("quality_score", -999) >= MIN_SCORE_ACCEPTABLE
    )

    selected_reasons = {
        reason
        for reason in str(
            selected.get("quality_reasons", "")
        ).split("|")
        if reason
    }
    needs_manual_review = bool(
        selected_reasons & SERIOUS_REVIEW_REASONS
    ) or selected.get("quality_flag") == "failed"

    selected_meta = selected.get("meta", {})
    selected_reasons = {
        reason for reason in str(
            selected.get("quality_reasons", "")
        ).split("|") if reason
    }
    serious_review_reasons = {
        "very_short_text",
        "replacement_characters",
        "possible_mojibake",
        "private_use_characters",
        "repeated_character_garbage",
        "low_language_plausibility",
    }
    needs_manual_review = bool(
        selected_reasons & serious_review_reasons
    ) or selected.get("quality_flag") == "failed"

    ranked_candidates = sorted(
        candidates,
        key=candidate_preference,
        reverse=True,
    )
    second_best_score = (
        ranked_candidates[1].get("quality_score")
        if len(ranked_candidates) > 1
        else None
    )
    selection_margin = (
        selected.get("quality_score") - second_best_score
        if second_best_score is not None
        else None
    )

    result = {
        **base,
        "clean_success": clean_success,
        "selected_method": selected.get("candidate_method"),
        # Keep legacy-compatible column name:
        "extraction_engine": selected.get("candidate_method"),
        "quality_score": selected.get("quality_score"),
        "quality_flag": selected.get("quality_flag"),
        "quality_reasons": selected.get("quality_reasons"),
        "needs_manual_review": needs_manual_review,
        "needs_manual_review": needs_manual_review,
        "n_chars_raw": selected.get("n_chars_raw"),
        "n_chars_clean": selected.get("n_chars_clean"),
        "n_pages": selected.get("n_pages"),
        "detected_language": selected.get("detected_language"),
        "language_score": selected.get("language_score"),
        "replacement_ratio": selected.get("replacement_ratio"),
        "mojibake_ratio": selected.get("mojibake_ratio"),
        "private_use_ratio": selected.get("private_use_ratio"),
        "repeated_character_ratio": selected.get("repeated_character_ratio"),
        "duplicate_line_ratio": selected.get("duplicate_line_ratio"),
        "one_char_token_ratio": selected.get("one_char_token_ratio"),
        "very_long_token_ratio": selected.get("very_long_token_ratio"),
        "no_vowel_token_ratio": selected.get("no_vowel_token_ratio"),
        "candidate_count": len(candidates),
        "candidate_methods": "|".join(c["candidate_method"] for c in candidates),
        "candidate_scores": score_listing,
        "second_best_score": second_best_score,
        "selection_margin": selection_margin,
        "selection_confidence": (
            "single_candidate"
            if selection_margin is None
            else "high"
            if selection_margin >= 5
            else "medium"
            if selection_margin >= 1
            else "low"
        ),
        "selection_reason": selection_reason,
        "selected_encoding": selected_meta.get("encoding", ""),
        "selected_encoding_source": selected_meta.get("encoding_source", ""),
        "selected_ftfy_applied": selected_meta.get("ftfy_applied", False),
        "mineru_used": selected.get("candidate_method") == "pdf_mineru",
        "candidate_errors": " | ".join(errors),
        "error": "" if clean_success else "selected_candidate_below_quality_threshold",
    }

    return result, candidate_rows


target_rows = work_manifest.to_dict("records")
results = []
candidate_results = []

for row in tqdm(target_rows, desc="Cleaning COM legacy files v7"):
    result, candidate_rows = process_one(row)
    results.append(result)
    candidate_results.extend(candidate_rows)

clean_file_manifest = pd.DataFrame(results)
candidate_manifest = pd.DataFrame(candidate_results)

essential_cols = [
    "document_id", "source_entry_id", "title", "decision_date",
    "case_numbers", "celex", "file_format",
    "download_success", "download_status", "raw_file_exists", "raw_file_path",
    "cleaning_target", "clean_success", "extraction_version",
    "quality_flag", "quality_score", "quality_reasons", "needs_manual_review",
    "n_chars_raw", "n_chars_clean", "n_pages",
    "selected_method", "extraction_engine", "mineru_used",
    "candidate_count", "candidate_methods", "candidate_scores",
    "second_best_score", "selection_margin", "selection_confidence",
    "selection_reason", "candidate_errors",
    "detected_language", "language_score",
    "replacement_ratio", "mojibake_ratio", "private_use_ratio",
    "repeated_character_ratio", "duplicate_line_ratio",
    "one_char_token_ratio", "very_long_token_ratio", "no_vowel_token_ratio",
    "selected_encoding", "selected_encoding_source", "selected_ftfy_applied",
    "clean_text_path", "readable_text_path", "markdown_path",
    "processed_at", "error",
]
essential_cols = [col for col in essential_cols if col in clean_file_manifest.columns]
clean_file_manifest = clean_file_manifest[essential_cols].copy()

clean_file_manifest.to_csv(CLEAN_FILE_MANIFEST_PATH, index=False, encoding="utf-8")
try:
    clean_file_manifest.to_excel(CLEAN_FILE_MANIFEST_XLSX_PATH, index=False)
except Exception as exc:
    print(f"Warning: could not write Excel manifest: {exc}")

if not candidate_manifest.empty:
    candidate_manifest.to_csv(
        CANDIDATE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )
elif not CANDIDATE_MANIFEST_PATH.exists():
    pd.DataFrame(columns=[
        "document_id",
        "candidate_method",
        "quality_score",
        "quality_flag",
    ]).to_csv(
        CANDIDATE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )

failed = clean_file_manifest[
    ~clean_file_manifest["clean_success"].fillna(False)
].copy()
failed.to_csv(FAILED_FILE_MANIFEST_PATH, index=False, encoding="utf-8")

print("Canonical clean-file manifest CSV:", CLEAN_FILE_MANIFEST_PATH)
print("Canonical clean-file manifest XLSX:", CLEAN_FILE_MANIFEST_XLSX_PATH)
print("Candidate manifest:", CANDIDATE_MANIFEST_PATH)
print("Best-quality files:", CLEAN_DATA_DIR)
print("Candidate audit files:", CANDIDATE_DATA_DIR)
print("Rows:", len(clean_file_manifest))
print("Clean success:", int(clean_file_manifest["clean_success"].fillna(False).sum()))
print("Failed/skipped:", len(failed))
display(clean_file_manifest.head(20))


Cleaning COM legacy files v7:   0%|          | 0/1505 [00:00<?, ?it/s]

Canonical clean-file manifest CSV: /home/edik/projects/eccjeu/output/com_legacy/com_legacy_clean_file_manifest.csv
Canonical clean-file manifest XLSX: /home/edik/projects/eccjeu/output/com_legacy/com_legacy_clean_file_manifest.xlsx
Candidate manifest: /home/edik/projects/eccjeu/output/com_legacy/com_legacy_extraction_candidate_manifest.csv
Best-quality files: /home/edik/projects/eccjeu/data/processed/com_legacy
Candidate audit files: /home/edik/projects/eccjeu/data/processed/com_legacy_candidates
Rows: 1505
Clean success: 1504
Failed/skipped: 1


,document_id,source_entry_id,title,decision_date,case_numbers,celex,file_format,download_success,download_status,raw_file_exists,...,very_long_token_ratio,no_vowel_token_ratio,selected_encoding,selected_encoding_source,selected_ftfy_applied,clean_text_path,readable_text_path,markdown_path,processed_at,error
0,com_legacy__31964D0599__html__f1021f1378,com_legacy_1964_line_2,Deca,22.10.1964,IV/71,31964D0599,html,True,already_downloaded,True,...,0.0,0.003024,utf-8,unicode_dammit,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:46,
1,com_legacy__31964D0599__pdf__1e622a12d6,com_legacy_1964_line_2,Deca,22.10.1964,IV/71,31964D0599,pdf,True,already_downloaded,True,...,0.0,0.001002,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:46,
2,com_legacy__31964D0566__html__81fa0f8340,com_legacy_1964_line_5,Grundig-Consten,23.09.1964,IV/3344; IV/4,31964D0566,html,True,already_downloaded,True,...,0.0,0.000950,utf-8,unicode_dammit,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:46,
3,com_legacy__31964D0566__pdf__df8a10f45c,com_legacy_1964_line_5,Grundig-Consten,23.09.1964,IV/3344; IV/4,31964D0566,pdf,True,already_downloaded,True,...,0.0,0.000359,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,
4,com_legacy__31964D0502__html__0e763dbecc,com_legacy_1964_line_8,Nicholas Freres + Vitapro,30.07.1964,IV/95,31964D0502,html,True,already_downloaded,True,...,0.0,0.002224,utf-8,unicode_dammit,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,
5,com_legacy__31964D0502__pdf__f50a290302,com_legacy_1964_line_8,Nicholas Freres + Vitapro,30.07.1964,IV/95,31964D0502,pdf,True,already_downloaded,True,...,0.0,0.000710,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,
6,com_legacy__31964D0344__html__9d411975d7,com_legacy_1964_line_11,Bendix + Mertens and Straat,01.06.1964,IV/12868,31964D0344,html,True,already_downloaded,True,...,0.0,0.001028,utf-8,unicode_dammit,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,
7,com_legacy__31964D0344__pdf__944f0fa8a5,com_legacy_1964_line_11,Bendix + Mertens and Straat,01.06.1964,IV/12868,31964D0344,pdf,True,already_downloaded,True,...,0.0,0.000000,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,
8,com_legacy__31964D0233__html__a25c04d101,com_legacy_1964_line_14,Grosfillex + Fillistorf,11.03.1964,IV/61,31964D0233,html,True,already_downloaded,True,...,0.0,0.002947,utf-8,unicode_dammit,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,
9,com_legacy__31964D0233__pdf__0f34c27802,com_legacy_1964_line_14,Grosfillex + Fillistorf,11.03.1964,IV/61,31964D0233,pdf,True,already_downloaded,True,...,0.0,0.000971,,,False,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,2026-08-31T14:57:47,


## 6. Quality summary and candidate comparison

In [15]:

if CLEAN_FILE_MANIFEST_PATH.exists():
    manifest = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)

    summary = (
        manifest.groupby(
            ["file_format", "quality_flag", "selected_method"],
            dropna=False,
        )
        .size()
        .reset_index(name="n")
        .sort_values(["file_format", "quality_flag", "selected_method"])
    )
    display(summary)

    review_cols = [
        "document_id", "title", "file_format",
        "selected_method", "quality_score", "quality_flag",
        "quality_reasons", "candidate_scores", "clean_text_path",
    ]
    review_cols = [c for c in review_cols if c in manifest.columns]

    display(
        manifest.sort_values(
            ["clean_success", "quality_score"],
            ascending=[True, True],
            na_position="first",
        )[review_cols].head(100)
    )
else:
    print("Run the processing cell first.")

if (
    CANDIDATE_MANIFEST_PATH.exists()
    and CANDIDATE_MANIFEST_PATH.stat().st_size > 0
):
    try:
        candidates = pd.read_csv(
            CANDIDATE_MANIFEST_PATH,
            low_memory=False,
        )
    except pd.errors.EmptyDataError:
        candidates = pd.DataFrame()
    if not candidates.empty:
        comparison = (
            candidates.groupby(
                ["file_format", "candidate_method", "quality_flag"],
                dropna=False,
            )
            .agg(
                n=("document_id", "size"),
                mean_score=("quality_score", "mean"),
                median_score=("quality_score", "median"),
                mean_chars=("n_chars_clean", "mean"),
            )
            .reset_index()
            .sort_values(["file_format", "mean_score"], ascending=[True, False])
        )
        display(comparison)


,file_format,quality_flag,selected_method,n
0,html,failed,html_best_container,1
1,html,fishy,html_best_container,2
2,html,ok,html_best_container,762
3,pdf,fishy,pdf_pymupdf_blocks_sorted,4
4,pdf,fishy,pdf_pymupdf_text_sorted,1
5,pdf,ok,pdf_pdftotext_layout,27
6,pdf,ok,pdf_pymupdf_blocks_sorted,270
7,pdf,ok,pdf_pymupdf_text_sorted,438


,document_id,title,file_format,selected_method,quality_score,quality_flag,quality_reasons,candidate_scores,clean_text_path
1474,com_legacy__31964D0233__html__b12d4e08fb,Grosfillex + Fillistorf,html,html_best_container,45.078,failed,very_short_text,html_best_container=45.08,/home/edik/projects/eccjeu/data/processed/com_...
1479,com_legacy__31969D0090__html__004426820d,EMO,html,html_best_container,38.205,ok,NaN,html_best_container=38.20,/home/edik/projects/eccjeu/data/processed/com_...
1475,com_legacy__31964D0233__html__49773e50ac,Grosfillex + Fillistorf,html,html_best_container,38.499,ok,NaN,html_best_container=38.50,/home/edik/projects/eccjeu/data/processed/com_...
1500,com_legacy__31996D0478__html__4877cbc7f4,ADALAT,html,html_best_container,38.594,fishy,many_duplicate_lines,html_best_container=38.59,/home/edik/projects/eccjeu/data/processed/com_...
1499,com_legacy__31995D0188__html__21b48c5a9b,COAPI,html,html_best_container,38.653,fishy,many_duplicate_lines,html_best_container=38.65,/home/edik/projects/eccjeu/data/processed/com_...
...,...,...,...,...,...,...,...,...,...
67,com_legacy__31970D0346__pdf__0fbf9f5ab1,ASBL Tube d'Acier Soudé Electr.,pdf,pdf_pymupdf_blocks_sorted,56.765,ok,NaN,pdf_pymupdf_blocks_sorted=56.77; pdf_pymupdf_t...,/home/edik/projects/eccjeu/data/processed/com_...
759,com_legacy__31985D0560__pdf__dd16af2f4d,BP/KELLOGG,pdf,pdf_pymupdf_blocks_sorted,56.916,ok,NaN,pdf_pymupdf_blocks_sorted=56.92; pdf_pdftotext...,/home/edik/projects/eccjeu/data/processed/com_...
1095,com_legacy__31990D0363__pdf__9e926e8c02,METALEUROP SA,pdf,pdf_pymupdf_blocks_sorted,56.952,ok,NaN,pdf_pymupdf_blocks_sorted=56.95; pdf_pymupdf_t...,/home/edik/projects/eccjeu/data/processed/com_...
1094,com_legacy__31990D0363__html__d2b916f247,METALEUROP SA,html,html_best_container,57.017,ok,NaN,html_best_container=57.02,/home/edik/projects/eccjeu/data/processed/com_...


,file_format,candidate_method,quality_flag,n,mean_score,median_score,mean_chars
2,html,html_best_container,ok,762,66.144911,69.2020,46811.888451
0,html,html_best_container,failed,1,45.078000,45.0780,62.000000
1,html,html_best_container,fishy,2,38.623500,38.6235,1905.500000
3,pdf,pdf_mineru,ok,2,69.018500,69.0185,79570.500000
7,pdf,pdf_pymupdf_blocks_sorted,fishy,5,67.641800,67.0560,160701.400000
11,pdf,pdf_pymupdf_text_sorted,ok,732,67.582656,69.1415,49976.001366
8,pdf,pdf_pymupdf_blocks_sorted,ok,733,67.567824,69.0880,50180.597544
5,pdf,pdf_pdftotext_layout,ok,735,67.556324,69.1040,50241.248980
4,pdf,pdf_pdftotext_layout,fishy,5,67.453400,66.8970,160179.200000
10,pdf,pdf_pymupdf_text_sorted,fishy,6,67.154833,66.9320,139339.833333


## 7. Inspect one canonical file and all of its candidates

In [16]:

DOCUMENT_ID_TO_VIEW = None
# Example:
# DOCUMENT_ID_TO_VIEW = "com_legacy__31982D0465__html__abc123..."

if DOCUMENT_ID_TO_VIEW:
    manifest = pd.read_csv(CLEAN_FILE_MANIFEST_PATH, low_memory=False)
    hit = manifest[manifest["document_id"].astype(str).eq(str(DOCUMENT_ID_TO_VIEW))]

    if hit.empty:
        print("Document ID not found.")
    else:
        record = hit.iloc[-1].to_dict()
        keys = [
            "document_id", "title", "celex", "file_format",
            "selected_method", "quality_score", "quality_flag",
            "quality_reasons", "candidate_scores", "clean_text_path",
            "readable_text_path",
        ]
        print(json.dumps(
            {key: record.get(key) for key in keys},
            indent=2,
            ensure_ascii=False,
        ))

        canonical_path = Path(record["clean_text_path"])
        print("\n--- CANONICAL REGEX/LLM VERSION ---\n")
        print(canonical_path.read_text(encoding="utf-8", errors="replace")[:7000])

        readable_path = Path(record["readable_text_path"])
        if readable_path.exists():
            print("\n--- READABLE VERSION ---\n")
            print(readable_path.read_text(encoding="utf-8", errors="replace")[:7000])

        if CANDIDATE_MANIFEST_PATH.exists() and CANDIDATE_MANIFEST_PATH.stat().st_size > 0:
            candidate_manifest = pd.read_csv(CANDIDATE_MANIFEST_PATH, low_memory=False)
            candidate_hit = candidate_manifest[
                candidate_manifest["document_id"].astype(str).eq(str(DOCUMENT_ID_TO_VIEW))
            ].sort_values("quality_score", ascending=False)
            display(candidate_hit)
else:
    print("Set DOCUMENT_ID_TO_VIEW to inspect a cleaned file.")


Set DOCUMENT_ID_TO_VIEW to inspect a cleaned file.
